# 实践 9.4  靶标—诱饵FDR与共享肽段的蛋白推断

《统计遗传与智能育种学》第九章 · 电子资源 `09d_proteomics_fdr_inference.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：09d_proteomics_fdr_inference.ipynb。实践为每张模拟谱图生成最佳target与decoy候选分数，先进行逐谱target-decoy competition，只保留胜者，再按分数阈值累计target/decoy数量并以带+1修正的比例估计FDR和q值。随后另设唯一肽段与共享肽段混合的蛋白—肽段映射，比较仅凭检出肽段即报告蛋白与要求唯一肽支持的两种简化规则。该练习不替代MaxQuant、FragPipe或DIA-NN等正式分析软件，用于说明PSM层FDR估计与蛋白推断的逻辑。

运行检查：重启内核并运行全部单元  
预期：输出4个分数阈值下的target/decoy胜者数与估计FDR；固定种子下q≤1%保留146个target PSM胜者、最低分数约5.455；展示shared peptide为何不能唯一支持多个同源蛋白。

## 输入、完整代码、实际输出与解释

输入与可复现条件：模拟谱图的最佳target/decoy候选分数与一组unique/shared peptide映射；SEED=904；先逐谱target-decoy competition，再累计FDR并计算q值。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=904
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 904


In [2]:
# 每张模拟谱图各有一个最佳target与decoy候选，先做逐谱竞争，只保留胜者。
n=1500; target_score=rng.normal(2.0,1,n); target_score[:300]+=3.5; decoy_score=rng.normal(2.0,1,n)
winner_is_target=target_score>=decoy_score; win_score=np.maximum(target_score,decoy_score); label=np.where(winner_is_target,'target','decoy')
order=np.argsort(win_score)[::-1]; score=win_score[order]; lab=label[order]; cumT=np.cumsum(lab=='target'); cumD=np.cumsum(lab=='decoy')
# +1修正使小计数区更保守；q值为当前及更宽松分数阈值FDR估计的最小值。
fdr=(cumD+1)/np.maximum(cumT,1); q=np.minimum.accumulate(fdr[::-1])[::-1]
rows=[]
for cutoff in [5.5,5.0,4.5,4.0]:
 sel=score>=cutoff; T=np.sum((lab=='target')&sel); D=np.sum((lab=='decoy')&sel); rows.append((cutoff,T,D,(D+1)/max(T,1)))
print(pd.DataFrame(rows,columns=['score_cutoff','target_winners','decoy_winners','FDR_est']).round(4).to_string(index=False))
keep=(lab=='target')&(q<=.01); threshold=float(score[keep].min()) if keep.any() else np.nan; print('target PSM winners at q<=1%',int(keep.sum()),'score threshold',round(threshold,3) if np.isfinite(threshold) else 'none')

mapping=pd.DataFrame([('pepA','P1',1),('pepB','P1',1),('pepC','P1',0),('pepC','P2',0),('pepD','P2',1),('pepE','P3',0),('pepE','P4',0),('pepF','P4',1)],columns=['peptide','protein','unique']); any_rule=mapping.groupby('protein').peptide.nunique(); unique_rule=mapping[mapping.unique==1].groupby('protein').peptide.nunique(); out=pd.DataFrame({'any_peptide':any_rule,'unique_peptide':unique_rule}).fillna(0).astype(int); out['report_with_unique_rule']=out.unique_peptide>=1; print(out.to_string())


 score_cutoff  target_winners  decoy_winners  FDR_est
          5.5             139              0   0.0072
          5.0             193              3   0.0207
          4.5             249             14   0.0602
          4.0             300             37   0.1267
target PSM winners at q<=1% 146 score threshold 5.455
         any_peptide  unique_peptide  report_with_unique_rule
protein                                                      
P1                 3               2                     True
P2                 2               1                     True
P3                 1               0                    False
P4                 2               1                     True


结果解释：阈值越宽松target与decoy胜者同时增加；q≤1%保留146个target PSM胜者，最低分数约5.455。蛋白层示例中P3只有共享肽、没有唯一肽段，因此在本示例规则下不单独报告；PSM层FDR控制并不会自动解决蛋白推断歧义。

可修改练习：去掉FDR估计中的+1修正后比较小样本阈值；再新增一个只属于P3的唯一肽段，观察蛋白报告规则如何变化。